In [2]:
print("Hello")

Hello


In [ ]:
from langchain import PromptTemplate
from langchain.chains import RetrievalQA
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Pinecone
import pinecone
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.prompts import PromptTemplate
from langchain.llms import CTransformers

ModuleNotFoundError: No module named 'langchain'

In [3]:
#Extract data from pdf
def load_pdf(data):
    loader = DirectoryLoader(data,
                    glob="*.pdf",
                    loader_cls=PyPDFLoader)
    
    documents = loader.load()

    return documents

In [4]:
import os
os.chdir(r"C:\Users\dharu\OneDrive\Desktop\Medical-Chatbot")


In [5]:
extracted_data = load_pdf("data/")


In [6]:
#extracted_data

In [12]:
#Create text chunks
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 20)
    text_chunks = text_splitter.split_documents(extracted_data)

    return text_chunks

In [13]:
text_chunks = text_split(extracted_data)
print("length of my chunk:", len(text_chunks))

length of my chunk: 5859


In [14]:
#text_chunks

In [15]:
#download embedding model
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [17]:
embeddings = download_hugging_face_embeddings()

In [18]:
#embeddings

In [19]:
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

Length 384


In [20]:
#query_result

In [23]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec

load_dotenv()

api_key = os.getenv("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)

index_name = "medical-chatbot"

if index_name not in [i.name for i in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"✅ Created index '{index_name}'")
else:
    print(f"ℹ️ Index '{index_name}' already exists")


✅ Created index 'medical-chatbot'


In [24]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
INDEX_NAME = "medical-bot"

# Create Pinecone client instance
pc = Pinecone(api_key=PINECONE_API_KEY)


In [25]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings,
)


In [26]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings,
)


In [27]:
retriver=docsearch.as_retriever(search_type="similarity",search_kwargs={"k":3})

In [28]:
retrived_docs=retriver.invoke("What is Fever")

In [29]:
retrived_docs

[Document(id='e8207633-db47-4284-a7b7-59965237d155', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 69.0, 'page_label': '70', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='fever in children. This disease is most often caused by\ntypes 3 and 7. Symptoms, which appear suddenly and\nusually disappear in less than a week, include:\n• inflammation of the lining of the eyelid (conjunctivitis)\n•f e v e r\n• sore throat (pharyngitis)\n• runny nose\n• inflammation of lymph glands in the neck (cervical\nadenitis)\nGALE ENCYCLOPEDIA OF MEDICINE 256\nAdenovirus infections\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 56'),
 Document(id='c29d4906-2ea3-46b4-8be8-df1c76fc36fd', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 60.0, 'page_label': '61', 'producer': 'PDFlib+PDI 5.0.0 (Su

In [30]:
from langchain import PromptTemplate
from langchain.chains import RetrievalQA
from langchain.llms import CTransformers

# Updated prompt with medical restriction + doctor advice
prompt_template = """
You are a helpful medical assistant.
Use the following pieces of information to answer the user's medical question.
If the question is not related to medical topics, respond with:
"Only ask questions related to medical topics."

Always recommend visiting a doctor in your answer, even if the answer is known.

Context: {context}
Question: {question}

Helpful answer:
"""

PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])
chain_type_kwargs = {"prompt": PROMPT}


In [32]:
llm = CTransformers(
    model="Models/llama-2-7b-chat.ggmlv3.q4_0.bin",
    model_type="llama",
    config={
        'max_new_tokens': 512,
        'temperature': 0.8
    }
)


In [33]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever(search_kwargs={'k': 2}),
    return_source_documents=True,
    chain_type_kwargs=chain_type_kwargs
)


In [40]:
prompt_template = """
You are a professional medical assistant.
Use ONLY the provided context to answer the user's medical question.
If the question is NOT related to medical topics, reply:
"Only medical-related questions are allowed."

Always recommend visiting a doctor at the end of your answer.

Context:
{context}

Question:
{question}

Helpful answer:
"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)
chain_type_kwargs = {"prompt": PROMPT}


In [41]:
llm = CTransformers(
    model="Models/llama-2-7b-chat.ggmlv3.q4_0.bin",  # Adjust if folder differs
    model_type="llama",
    config={
        'max_new_tokens': 256,      # Prevents overly long answers
        'context_length': 512,      # Avoids overflow errors
        'temperature': 0.2,         # Lower = more factual
        'repetition_penalty': 1.2   # Avoids repeating words
    }
)


In [42]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever(search_kwargs={'k': 1}),  # 1 chunk for precision
    return_source_documents=True,
    chain_type_kwargs=chain_type_kwargs
)


In [43]:
while True:
    user_input=input(f"Input Prompt:")
    result=qa({"query": user_input})
    print("Response : ", result["result"])

Response :  Fever is a symptom of an underlying medical condition. It's important to consult with a doctor if you suspect that your child has a fever, as it can be caused by a variety of things such as infection, allergy or other illnesses. A proper diagnosis and treatment plan can help alleviate the symptoms and prevent complications.
Please visit a doctor for further evaluation and appropriate medical advice.
Response :  Acne is a common skin condition characterized by the presence of comedones (blackheads and whiteheads), pustules, nodules, and inflamed red bumps on the face, chest, back, and other areas of the body. It can be caused by hormonal changes, genetics, environmental factors such as excessive oil production, clogged pores, and bacterial infections. Acne can lead to emotional distress, low self-esteem, and social isolation if left untreated. Treatment options include topical creams or gels containing benzoyl peroxide or salicylic acid, oral antibiotics, retinoids, and horm

Number of tokens (513) exceeded maximum context length (512).
Number of tokens (514) exceeded maximum context length (512).
Number of tokens (515) exceeded maximum context length (512).
Number of tokens (516) exceeded maximum context length (512).
Number of tokens (517) exceeded maximum context length (512).


Response :  Thank you for reaching out! As a medical assistant, I must inform you that only medical-related questions are allowed on this platform. However, I can provide some general information about how to prevent colds and other infections.

To prevent the spread of colds and other viral infections, it is important to practice good hygiene habits such as:

* Washing your hands frequently with soap and water, especially after blowing your nose, coughing or sneezing
* Avoiding close contact with people who are sick
* Covering your mouth and nose when you cough or sneeze
* Cleaning surfaces and objects that may be contaminated with the virus

It is also recommended to get enough sleep, exercise regularly, manage stress levels, and eat a healthy diet rich in fruits, vegetables, and whole grains. These habits can help boost your immune system and reduce the risk of getting sick.

However, if you are experiencing persistent or severe cold symptoms, it is important to consult with a medic

KeyboardInterrupt: 